# 🦾 Desktop AI Robot Arm: 3D 6-DOF Reinforcement Learning (PPO)
### Master 3D Robotic Manipulator Control in MuJoCo on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

---

## 🎯 What You Will Learn in This Notebook
1. **3D 6-DOF Robot Arm Kinematics**: How 6 articulated joints (Base Yaw, Shoulder Pitch, Elbow Pitch, Forearm Roll, Wrist Pitch, Tool Roll) enable a desktop robot arm to reach any point $(X, Y, Z)$ in 3D space.
2. **Google DeepMind's MuJoCo**: The gold-standard physics engine used by DeepMind and top robotics labs for simulated physics.
3. **Reinforcement Learning (RL) with PPO**: Training a neural network policy from scratch through trial-and-error using **Proximal Policy Optimization**.
4. **Simulation-to-Video Rendering**: Recording and playing back MP4 videos of your trained robot arm directly inside Colab.
5. **Interactive 3D Goal Targeting**: Giving the trained arm custom 3D spatial coordinates and watching it navigate smoothly in real-time.

## ⚙️ Step 1: Install Dependencies & Setup Headless GPU Rendering

> **Colab Tip**: For maximum speed, go to **Runtime > Change runtime type** and select **T4 GPU** (or standard CPU, both work!).
We configure `MUJOCO_GL=egl` to enable GPU-accelerated headless 3D rendering.

In [ ]:
# Install Gymnasium, DeepMind MuJoCo, Stable-Baselines3, and video encoders
!pip install -q gymnasium mujoco stable-baselines3 imageio[ffmpeg] matplotlib

import os
import numpy as np
import matplotlib.pyplot as plt
import imageio
from IPython.display import Video, display

# Enable GPU headless rendering
os.environ["MUJOCO_GL"] = "egl"

import gymnasium as gym
from gymnasium import spaces
import mujoco
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

print("✅ All dependencies installed and MuJoCo EGL rendering configured successfully!")

## 📐 Step 2: Define the 3D 6-DOF Desktop Robot Arm Environment

Below, we define a complete **MuJoCo MJCF (XML)** model representing a real desktop articulated robot arm (similar to open-source arms like the SO-100 or Koch arm):

| Joint | Name | Axis | Motion Range | Real-World Equivalent |
|---|---|---|---|---|
| **Joint 1** | Base Yaw | Z-axis | $-180^\circ$ to $+180^\circ$ | Waist rotating turret servo |
| **Joint 2** | Shoulder Pitch | Y-axis | $-90^\circ$ to $+90^\circ$ | Main lift arm servo |
| **Joint 3** | Elbow Pitch | Y-axis | $-120^\circ$ to $+120^\circ$ | Forearm angle servo |
| **Joint 4** | Forearm Roll | Z-axis | $-180^\circ$ to $+180^\circ$ | Arm axial twist servo |
| **Joint 5** | Wrist Pitch | Y-axis | $-90^\circ$ to $+90^\circ$ | Wrist elevation servo |
| **Joint 6** | Tool Roll | Z-axis | $-180^\circ$ to $+180^\circ$ | Gripper orientation servo |

- **Actuators**: Position-controlled actuators with damping, simulating digital smart servos (e.g. Feetech STS3215 or Dynamixel).
- **Goal**: Move the green end-effector site (`ee_site`) to touch the red target sphere positioned randomly in 3D space above the desk.

In [ ]:
DESKTOP_ARM_6DOF_XML = """
<mujoco model="desktop_arm_6dof">
  <compiler angle="radian" coordinate="local" inertiafromgeom="true"/>
  <option timestep="0.005" gravity="0 0 -9.81" integrator="Euler"/>
  
  <default>
    <joint damping="0.8" armature="0.01"/>
    <geom contype="1" conaffinity="1" friction="0.8 0.1 0.1"/>
    <position kp="40" kv="5"/>
  </default>

  <asset>
    <texture type="skybox" builtin="gradient" rgb1="0.3 0.5 0.7" rgb2="0.1 0.1 0.2" width="512" height="512"/>
    <texture name="grid" type="2d" builtin="checker" rgb1="0.85 0.85 0.85" rgb2="0.65 0.65 0.65" width="512" height="512"/>
    <material name="grid_mat" texture="grid" texrepeat="5 5" reflectance="0.1"/>
    <material name="arm_mat" rgba="0.15 0.45 0.85 1.0" specular="0.4" shininess="0.3"/>
    <material name="joint_mat" rgba="0.95 0.55 0.1 1.0" specular="0.5" shininess="0.4"/>
    <material name="flange_mat" rgba="0.25 0.25 0.28 1.0" specular="0.2"/>
    <material name="target_mat" rgba="0.95 0.15 0.15 0.85" specular="0.6" shininess="0.6"/>
    <material name="table_mat" rgba="0.9 0.9 0.92 1.0" specular="0.2"/>
    <material name="ee_mat" rgba="0.1 0.85 0.3 1.0" specular="0.5"/>
  </asset>

  <worldbody>
    <light pos="0.2 -0.3 1.5" dir="-0.2 0.3 -1.0" diffuse="0.85 0.85 0.85" specular="0.3 0.3 0.3"/>
    <light pos="-0.5 0.5 1.2" dir="0.5 -0.5 -1.0" diffuse="0.4 0.4 0.4"/>

    <geom name="floor" type="plane" size="1.2 1.2 0.05" material="grid_mat"/>

    <body name="table" pos="0 0 0.15">
      <geom name="table_top" type="box" size="0.35 0.35 0.015" material="table_mat"/>
      <geom name="table_leg1" type="cylinder" size="0.02 0.07" pos="0.3 0.3 -0.075" material="flange_mat"/>
      <geom name="table_leg2" type="cylinder" size="0.02 0.07" pos="-0.3 0.3 -0.075" material="flange_mat"/>
      <geom name="table_leg3" type="cylinder" size="0.02 0.07" pos="0.3 -0.3 -0.075" material="flange_mat"/>
      <geom name="table_leg4" type="cylinder" size="0.02 0.07" pos="-0.3 -0.3 -0.075" material="flange_mat"/>
    </body>

    <body name="target" pos="0.15 0.10 0.32" mocap="true">
      <geom name="target_geom" type="sphere" size="0.025" material="target_mat" contype="0" conaffinity="0"/>
      <site name="target_site" pos="0 0 0" size="0.008" rgba="1 1 1 0.9"/>
    </body>

    <body name="arm_base" pos="0 0 0.165">
      <geom name="base_plate" type="cylinder" size="0.065 0.015" material="flange_mat"/>
      <geom name="base_pillar" type="cylinder" size="0.045 0.03" pos="0 0 0.03" material="arm_mat"/>

      <body name="link1" pos="0 0 0.06">
        <joint name="joint1_base_yaw" type="hinge" axis="0 0 1" range="-3.1416 3.1416"/>
        <geom name="j1_cylinder" type="cylinder" size="0.04 0.025" material="joint_mat"/>

        <body name="link2" pos="0 0 0.04">
          <joint name="joint2_shoulder_pitch" type="hinge" axis="0 1 0" range="-1.5708 1.5708"/>
          <geom name="j2_hub" type="cylinder" size="0.03 0.025" fromto="0 -0.025 0 0 0.025 0" material="joint_mat"/>
          <geom name="upper_arm" type="capsule" fromto="0 0 0 0 0 0.14" size="0.022" material="arm_mat"/>

          <body name="link3" pos="0 0 0.14">
            <joint name="joint3_elbow_pitch" type="hinge" axis="0 1 0" range="-2.0944 2.0944"/>
            <geom name="j3_hub" type="cylinder" size="0.026 0.022" fromto="0 -0.022 0 0 0.022 0" material="joint_mat"/>
            <geom name="forearm" type="capsule" fromto="0 0 0 0 0 0.12" size="0.018" material="arm_mat"/>

            <body name="link4" pos="0 0 0.12">
              <joint name="joint4_wrist_roll" type="hinge" axis="0 0 1" range="-3.1416 3.1416"/>
              <geom name="j4_swivel" type="cylinder" size="0.018 0.015" pos="0 0 0.015" material="joint_mat"/>

              <body name="link5" pos="0 0 0.03">
                <joint name="joint5_wrist_pitch" type="hinge" axis="0 1 0" range="-1.5708 1.5708"/>
                <geom name="j5_hub" type="sphere" size="0.018" material="joint_mat"/>
                <geom name="wrist_link" type="capsule" fromto="0 0 0 0 0 0.035" size="0.014" material="arm_mat"/>

                <body name="link6_flange" pos="0 0 0.035">
                  <joint name="joint6_tool_roll" type="hinge" axis="0 0 1" range="-3.1416 3.1416"/>
                  <geom name="tool_disc" type="cylinder" size="0.02 0.008" pos="0 0 0.008" material="flange_mat"/>

                  <geom name="gripper_base" type="box" size="0.025 0.008 0.006" pos="0 0 0.018" material="flange_mat"/>
                  <geom name="finger_left" type="box" size="0.004 0.008 0.018" pos="-0.016 0 0.032" material="joint_mat"/>
                  <geom name="finger_right" type="box" size="0.004 0.008 0.018" pos="0.016 0 0.032" material="joint_mat"/>

                  <site name="ee_site" pos="0 0 0.045" size="0.008" rgba="0.1 0.9 0.3 0.9"/>
                </body>
              </body>
            </body>
          </body>
        </body>
      </body>
    </body>

    <camera name="cam_isometric" pos="0.55 -0.55 0.65" xyaxes="0.707 0.707 0 -0.408 0.408 0.816"/>
    <camera name="cam_front" pos="0 -0.65 0.45" xyaxes="1 0 0 0 0.45 0.893"/>
    <camera name="cam_overhead" pos="0 0 0.9" xyaxes="1 0 0 0 1 0"/>
  </worldbody>

  <actuator>
    <position name="act_j1" joint="joint1_base_yaw" kp="50" ctrlrange="-3.1416 3.1416"/>
    <position name="act_j2" joint="joint2_shoulder_pitch" kp="60" ctrlrange="-1.5708 1.5708"/>
    <position name="act_j3" joint="joint3_elbow_pitch" kp="50" ctrlrange="-2.0944 2.0944"/>
    <position name="act_j4" joint="joint4_wrist_roll" kp="30" ctrlrange="-3.1416 3.1416"/>
    <position name="act_j5" joint="joint5_wrist_pitch" kp="25" ctrlrange="-1.5708 1.5708"/>
    <position name="act_j6" joint="joint6_tool_roll" kp="20" ctrlrange="-3.1416 3.1416"/>
  </actuator>
</mujoco>
"""

class DesktopArm6DOFEnv(gym.Env):
    metadata = {"render_modes": ["rgb_array"], "render_fps": 40}

    def __init__(self, render_mode="rgb_array", max_episode_steps=150, camera_name="cam_isometric"):
        super().__init__()
        self.model = mujoco.MjModel.from_xml_string(DESKTOP_ARM_6DOF_XML.strip())
        self.data = mujoco.MjData(self.model)

        self.render_mode = render_mode
        self.camera_name = camera_name
        self.max_episode_steps = max_episode_steps
        self.step_count = 0
        self.frame_skip = 5
        self.action_scale = 0.05

        self.ee_site_id = mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_SITE, "ee_site")
        self.target_mocap_id = 0

        self.ctrl_min = self.model.actuator_ctrlrange[:, 0]
        self.ctrl_max = self.model.actuator_ctrlrange[:, 1]

        # Action Space: 6 continuous motor delta commands
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(6,), dtype=np.float32)

        # Observation Space: 27 continuous values
        # cos(qpos[0..5]) (6) + sin(qpos[0..5]) (6) + qvel[0..5] (6) + ee_pos (3) + target_pos (3) + delta_pos (3)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(27,), dtype=np.float32)

        self.renderer = None
        self.target_pos = np.array([0.15, 0.10, 0.32], dtype=np.float32)

    def _get_obs(self):
        qpos = self.data.qpos[:6].copy()
        qvel = self.data.qvel[:6].copy()
        ee_pos = self.data.site_xpos[self.ee_site_id].copy()
        delta_pos = self.target_pos - ee_pos

        return np.concatenate([
            np.cos(qpos), np.sin(qpos),
            qvel,
            ee_pos, self.target_pos, delta_pos
        ]).astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.step_count = 0

        mujoco.mj_resetData(self.model, self.data)

        # Home pose with slight elbow and shoulder elevation
        home_qpos = np.array([0.0, 0.35, 0.55, 0.0, 0.2, 0.0])
        noise = self.np_random.uniform(-0.05, 0.05, size=6)
        self.data.qpos[:6] = home_qpos + noise
        self.data.qvel[:6] = 0.0
        self.data.ctrl[:6] = self.data.qpos[:6]

        # Sample reachable 3D target coordinates
        r = self.np_random.uniform(0.12, 0.28)
        theta = self.np_random.uniform(-np.pi * 0.75, np.pi * 0.75)
        z = self.np_random.uniform(0.20, 0.38)

        self.target_pos = np.array([r * np.cos(theta), r * np.sin(theta), z], dtype=np.float32)
        self.data.mocap_pos[self.target_mocap_id] = self.target_pos
        mujoco.mj_forward(self.model, self.data)

        obs = self._get_obs()
        return obs, {"target_pos": self.target_pos.copy()}

    def step(self, action):
        self.step_count += 1
        action = np.clip(action, -1.0, 1.0)

        target_ctrl = self.data.ctrl[:6] + (action * self.action_scale)
        self.data.ctrl[:6] = np.clip(target_ctrl, self.ctrl_min, self.ctrl_max)

        for _ in range(self.frame_skip):
            mujoco.mj_step(self.model, self.data)

        ee_pos = self.data.site_xpos[self.ee_site_id]
        dist = np.linalg.norm(ee_pos - self.target_pos)

        # Reward design: Distance penalty + smoothness penalty + precision bonus
        reward_dist = -dist
        reward_action = -0.005 * np.sum(np.square(action))
        reward_bonus = 2.0 if dist < 0.03 else (0.5 if dist < 0.06 else 0.0)

        reward = float(reward_dist + reward_action + reward_bonus)
        terminated = False
        truncated = self.step_count >= self.max_episode_steps

        obs = self._get_obs()
        info = {"distance": float(dist), "is_success": bool(dist < 0.03)}
        return obs, reward, terminated, truncated, info

    def render(self):
        if self.render_mode != "rgb_array":
            return None
        if self.renderer is None:
            self.renderer = mujoco.Renderer(self.model, height=480, width=640)
        self.renderer.update_scene(self.data, camera=self.camera_name)
        return self.renderer.render()

    def close(self):
        if self.renderer is not None:
            self.renderer.close()
            self.renderer = None

print("✅ 3D 6-DOF Robot Arm Environment defined successfully!")

## 🔍 Step 3: Inspect Environment Dimensions
Let's create an instance of the environment and inspect the observation and action spaces.

In [ ]:
env = DesktopArm6DOFEnv(render_mode="rgb_array")
obs, info = env.reset()

print(f"Observation Space Shape: {env.observation_space.shape} (27 continuous state values)")
print(f"Action Space Shape:      {env.action_space.shape} (6 continuous motor commands)")
print(f"Initial Target Position: {info['target_pos']}")
print(f"Initial Observation Sample (first 6 values = cos of angles): {obs[:6]}")

## 🎬 Step 4: Record Baseline Video (Untrained Random Policy)
Before training any AI, let's see how the robot behaves when choosing completely random motor actions.
Notice that the arm flails randomly and fails to reach the red target sphere.

In [ ]:
print("🎥 Recording 150 steps of untrained random actions...")
frames = []
obs, _ = env.reset(seed=42)

for _ in range(150):
    random_action = env.action_space.sample()  # Pure random actions
    obs, reward, terminated, truncated, info = env.step(random_action)
    frame = env.render()
    frames.append(frame)
    if terminated or truncated:
        break

untrained_video_path = "arm_untrained_baseline.mp4"
imageio.mimsave(untrained_video_path, frames, fps=30)
print(f"Video saved to: {untrained_video_path}")

# Play video directly inside Colab
Video(untrained_video_path, embed=True, width=640)

## 🧠 Step 5: Train with Reinforcement Learning (PPO)

### What is PPO (Proximal Policy Optimization)?
- **Actor Network**: Learns the mapping from state $(27 \text{ values}) \to$ motor movements $(6 \text{ actions})$.
- **Critic Network**: Predicts how much cumulative reward the arm will collect from the current position.
- **PPO Clipping**: Prevents the neural network from taking overly aggressive update steps, ensuring smooth and stable learning.

Let's train for **150,000 timesteps** (~3–5 minutes on Colab GPU).

In [ ]:
print("=" * 60)
print("🚀 Initializing PPO Agent for 3D 6-DOF Robot Arm...")
print("=" * 60)

train_env = DesktopArm6DOFEnv(render_mode=None)  # Headless mode for max training speed

model = PPO(
    policy="MlpPolicy",
    env=train_env,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.0,
    verbose=1,
    tensorboard_log="./ppo_desktop_arm_tb/"
)

TOTAL_TIMESTEPS = 150_000
print(f"\nTraining for {TOTAL_TIMESTEPS:,} timesteps...")
model.learn(total_timesteps=TOTAL_TIMESTEPS, progress_bar=True)

# Save the trained neural network model
model_path = "desktop_arm_6dof_ppo.zip"
model.save(model_path)
print(f"\n💾 Model successfully saved as: {model_path}")

## 📊 Step 6: Evaluate the Trained Policy
Let's evaluate the trained policy across 15 unseen random target coordinates.

In [ ]:
print("Evaluating policy performance...")
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=15)
print(f"🏆 Evaluation across 15 episodes: Mean Reward = {mean_reward:.2f} +/- {std_reward:.2f}")

# Record end-effector tracking distances
distances = []
for episode in range(10):
    obs, _ = env.reset()
    for step in range(150):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        if step == 149:
            distances.append(info["distance"] * 100)  # in cm

plt.figure(figsize=(8, 4))
plt.bar(range(1, 11), distances, color="#3b82f6", edgecolor="#1e40af")
plt.axhline(3.0, color="red", linestyle="--", label="Success Threshold (3 cm)")
plt.xlabel("Episode")
plt.ylabel("Final Distance to Target (cm)")
plt.title("Trained 6-DOF Arm Final Distance to 3D Targets")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 🎥 Step 7: Render Video of Trained 3D 6-DOF Robot Arm in Action!
Now we render the trained policy navigating the 6-DOF arm smoothly to reach 3D spatial targets.

In [ ]:
print("🎥 Generating evaluation video with trained policy...")
trained_frames = []
obs, info = env.reset(seed=101)

# Run for 300 steps across multiple target goals
for step in range(300):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    frame = env.render()
    trained_frames.append(frame)

    if terminated or truncated:
        obs, info = env.reset()

trained_video_path = "trained_6dof_arm_demo.mp4"
imageio.mimsave(trained_video_path, trained_frames, fps=30)
print(f"Trained video saved to: {trained_video_path}")

# Display video in notebook
Video(trained_video_path, embed=True, width=640)

## 🎯 Step 8: Interactive Test — Command Custom 3D Coordinates!
Try typing your own $(X, Y, Z)$ target coordinates and watch the policy solve the inverse kinematics dynamically in 3D simulation!

In [ ]:
# Set your custom 3D coordinate target (in meters from robot base)
# Reachable workspace: X within [-0.25, 0.25], Y within [-0.25, 0.25], Z within [0.20, 0.38]
custom_x = 0.18
custom_y = 0.12
custom_z = 0.30

print(f"🎯 Commanding arm to custom target: X={custom_x}, Y={custom_y}, Z={custom_z}")

obs, _ = env.reset()
# Override target position
env.target_pos = np.array([custom_x, custom_y, custom_z], dtype=np.float32)
env.data.mocap_pos[env.target_mocap_id] = env.target_pos
mujoco.mj_forward(env.model, env.data)
obs = env._get_obs()

custom_frames = []
for step in range(120):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    custom_frames.append(env.render())

final_distance = info["distance"]
print(f"Final End-Effector Distance: {final_distance * 100:.2f} cm (Success: {info['is_success']})")

custom_video_path = "custom_target_test.mp4"
imageio.mimsave(custom_video_path, custom_frames, fps=30)
Video(custom_video_path, embed=True, width=640)

## 🚀 Next Steps: Connecting Gemini Vision & Moving to Hardware

Now that your 3D 6-DOF policy is trained:
1. **Gemini 2.0 Flash Vision**: In `gemini_vision_planner.py`, camera frames from this 3D simulator are passed to Google's Gemini Vision model to automatically identify target objects and extract their 3D coordinates.
2. **Sim-to-Real**: When you are ready to assemble physical hardware (e.g. 3D printing a 6-DOF arm with Feetech servos), you can export the policy or map the learned joint angles directly to your ESP32 or Raspberry Pi servo controller via serial protocol!